# setup

In [ ]:
!pip install -U transformers
!pip install -q -U peft datasets bitsandbytes trl
!pip uninstall -y pyarrow
!pip install -U --no-cache-di pyarrow

# inference

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

device = "cuda"

model_id = "Qwen/Qwen3.5-0.8B"

model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device, dtype= torch.bfloat16)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
prompt = """
          You are extracting financial-statement annotations from a single page image into the exact Pydantic schema `PageExtraction`.

          Return ONLY valid JSON.
          Do NOT return markdown, code fences, comments, prose, or extra keys.

          Top-level schema (exact keys only):
          {
            "meta": {
              "entity_name": "<string or null>",
              "page_num": "<string or null>",
              "type": "balance_sheet|income_statement|cash_flow|notes|contents_page|title|declaration|profits|other",
              "title": "<string or null>"
            },
            "facts": [
              {
                "bbox": { "x": <number>, "y": <number>, "w": <number>, "h": <number> },
                "value": "<string>",
                "note": "<string or null>",
                "is_beur": <true|false|null>,
                "beur_num": "<string or null>",
                "refference": "<string>",
                "date": "<string or null>",
                "path": ["<string>", "..."],
                "currency": "ILS|USD|EUR|GBP|null",
                "scale": 1|1000|1000000|null,
                "value_type": "amount|%|null"
              }
            ]
          }

          Type/validation rules (must match schema):
          1. `meta` and `facts` are required.
          2. `refference` is required string (keep exact misspelling). Use "" when missing.
          3. `path` is required list of strings (use [] when unknown, never null).
          4. `is_beur` must be boolean or null (never "true"/"false" strings).
          5. Use JSON null literal for missing optional values (never "null" string).
          6. Do not include any keys not listed above.

          Extraction rules:
          1. Extract all visible numeric/table facts, including negatives in parentheses and totals.
          2. Also extract placeholder values ("-", "—", "–", similar dashes) when they appear as actual cell values.
          3. Keep `value` exactly as printed (commas, parentheses, percent sign, dash placeholders).
          4. `bbox` must tightly cover the value text only, in pixel coordinates of the original image.
          5. `note`: textual qualifier tied to the specific fact (for example `*without debt insurance`), else null.
          6. `beur_num`: use when this fact belongs to data presented inside that beur section; else null.
          7. `refference`: use when this fact points/references another beur; else "".
          8. `currency`: infer from page/header context (for example "שקלים חדשים" -> "ILS"), else null.
          9. `scale`: 1 unless page/header indicates thousands or millions.
          10. `value_type`: "amount" unless the fact is a percentage ("%").
          11. `date`: map by column/year context when explicit and unambiguous; else null.
          12. `meta.type`: use "profits" for "דוחות על הפעילויות"; otherwise choose the best enum value.
          13. No duplicate facts. Order `facts` top-to-bottom, then left-to-right.
          14. Output UTF-8 Hebrew directly (do not escape to unicode sequences).

          Path hierarchy rule:
          1. Build `path` with horizontal-axis hierarchy first (row-side structure: group -> subgroup -> line item).
          2. Then append vertical-axis hierarchy only when it adds business semantics not already represented by dedicated fields.
          3. Do not encode period/currency/scale/value-format in `path` when already represented by `date`/`currency`/`scale`/`value_type`.
          4. Keep labels exactly as shown. Do not invent levels.
"""

In [ ]:
import torch
from transformers import TextStreamer

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "/content/page_0004.png",
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# Prepare inputs
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,

)
inputs = inputs.to(model.device)

# Streamer (prints as it generates)
streamer = TextStreamer(
    tokenizer=processor.tokenizer if hasattr(processor, "tokenizer") else processor.tokenizer,
    skip_prompt=True,
    skip_special_tokens=True,
)

# Generate with streaming and specified parameters
_ = model.generate(
    **inputs,
    max_new_tokens=4096,
    streamer=streamer,
    do_sample=True, # Required for temperature, top_p, top_k to take effect
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
)


# Fine Tuning

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

device = "cuda"

model_id = "Qwen/Qwen3.5-0.8B"

model = AutoModelForCausalLM.from_pretrained(model_id, device_map=device, dtype= torch.bfloat16)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=2,                     # rank (16 = strong baseline)
    lora_alpha=4,            # usually 2x r
    lora_dropout=0.05,        # small regularization
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,
)

model = get_peft_model(model, peft_config).to(device)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("asafd60/FineTree-annotated-pages-no-bbox")

In [ ]:
dataset['train'][0]['image'].size

In [ ]:
system_message = "You are an helpfull assistant"


def format_data(sample):
    return [
    {
        "role": "system",
        "content": system_message,
    },
    {
        "role": "user",
        "content": [
          {
              "type": "image",
              "image": sample["image"]
          },
          {
              "type": "text",
              "text": prompt
          }
          ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "type":"text", "text": sample["text"]
            }
          ]
    }]

# train_data = [format_data(sample) for sample in dataset["train"]]
# validation_data = [format_data(sample) for sample in dataset["validation"]]

In [ ]:
def text_generator(sample_data):
    messages = processor.apply_chat_template(
        sample_data,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    image = sample_data[1]["content"][0]["image"]
    inputs = processor(text=[messages], images= image, add_special_tokens=False, return_tensors="pt").to("cuda")

    # Explicitly select kwargs for generation, excluding pixel_values as it's causing an error
    # It appears pixel_values are handled internally by the model after processor prepares inputs.
    generate_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
    }

    generated_ids = modelƒ.generate(**generate_kwargs, max_new_tokens=4096)
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    ground_truth = sample_data[-1]["content"][0]["text"]
    del inputs
    del generated_ids
    return output_text, ground_truth

print(text_generator(train_data[0]))

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


('system\nYou are an helpfull assistant\nuser\n\n          You are extracting financial-statement annotations from a single page image into the exact Pydantic schema `PageExtraction`.\n\n          Return ONLY valid JSON.\n          Do NOT return markdown, code fences, comments, prose, or extra keys.\n\n          Top-level schema (exact keys only):\n          {\n            "meta": {\n              "entity_name": "<string or null>",\n              "page_num": "<string or null>",\n              "type": "balance_sheet|income_statement|cash_flow|notes|contents_page|title|declaration|profits|other",\n              "title": "<string or null>"\n            },\n            "facts": [\n              {\n                "bbox": { "x": <number>, "y": <number>, "w": <number>, "h": <number> },\n                "value": "<string>",\n                "note": "<string or null>",\n                "is_beur": <true|false|null>,\n                "beur_num": "<string or null>",\n                "refference":

In [ ]:
collate_sample = [dataset["train"][0], dataset["train"][1]] # Corrected to use raw dataset samples

def collate_fn(examples):
    formatted_conversations = [format_data(example) for example in examples]

    texts = [processor.apply_chat_template(conversation, tokenize=False) for conversation in formatted_conversations]
    image_inputs = [example["image"] for example in examples] # Extract images from the raw examples

    batch = processor(
        text=texts, images=image_inputs, return_tensors="pt", padding=True
    )
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = batch["input_ids"]

    return batch

collated_data = collate_fn(collate_sample)
print(collated_data.keys())  # dict_keys(['input_ids', 'attention_mask', 'pixel_values', 'labels'])

KeysView({'input_ids': tensor([[248045,   8678,    198,  ..., 248046,    198, 248044],
        [248045,   8678,    198,  ...,     92, 248046,    198]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 0],
        [1, 1, 1,  ..., 1, 1, 1]]), 'pixel_values': tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]]), 'image_grid_thw': tensor([[  1, 146, 104],
        [  1, 146, 104]]), 'labels': tensor([[248045,   8678,    198,  ..., 248046,    198, 248044],
        [248045,   8678,    198,  ...,     92, 248046,    198]])})


In [ ]:
from transformers import TrainingArguments
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=f"{model_id}-finetree",
    learning_rate=1e-3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_steps=5,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    save_total_limit=3,
    logging_steps=1,
    eval_steps = 5,
    remove_unused_columns=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collate_fn,

)

trainer.train()

Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 29.18 GiB. GPU 0 has a total capacity of 39.49 GiB of which 16.00 GiB is free. Including non-PyTorch memory, this process has 23.48 GiB memory in use. Of the allocated memory 16.76 GiB is allocated by PyTorch, and 6.22 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)